# Survey: All h11=2 Calabi-Yau threefolds

This notebook runs the full EKC construction on all 36 polytopes in the
CYTools database with $h^{1,1} = 2$, collecting statistics on phases,
contraction types, Coxeter groups, and cone generators.

Uses the adaptive GV degree strategy (start at `max_deg=4`, bump if needed).

In [ ]:
import cybir
from cybir import CYBirationalClass, patch_cytools
from cybir.core.types import ContractionType
import cytools
import numpy as np
from collections import Counter
import time

patch_cytools()

## Load all h11=2 polytopes

In [ ]:
polys = cytools.fetch_polytopes(h11=2, lattice="N")
print(f"Loaded {len(polys)} polytopes with h11=2")

## Run EKC construction on all polytopes

For each polytope, construct the birational class (fundamental domain),
then apply the Coxeter orbit expansion.

In [ ]:
results = []

for i, p in enumerate(polys):
    cy = p.triangulate().get_cy()
    
    t0 = time.perf_counter()
    ekc = CYBirationalClass.from_gv(cy, max_deg=4, verbose=False)
    n_fund = len(ekc.phases)
    
    ekc.apply_coxeter_orbit()
    n_total = len(ekc.phases)
    elapsed = time.perf_counter() - t0
    
    # Collect contraction type counts
    type_counts = Counter(
        c.contraction_type.display_name() for c in ekc.contractions
    )
    
    results.append({
        "idx": i,
        "ekc": ekc,
        "n_fund": n_fund,
        "n_total": n_total,
        "n_contractions": len(ekc.contractions),
        "type_counts": type_counts,
        "n_sym_flops": len(ekc._sym_flop_pairs),
        "coxeter_type": ekc.coxeter_type,
        "coxeter_order": ekc.coxeter_order,
        "n_inf_gens": len(ekc.infinity_cone_gens),
        "n_eff_gens": len(ekc.eff_cone_gens),
        "time": elapsed,
    })
    
    print(f"  #{i}: {elapsed:.2f}s  fund={n_fund} total={n_total}  "
          f"sym_flops={len(ekc._sym_flop_pairs)}  "
          f"coxeter={ekc.coxeter_type}")

## Summary statistics

In [ ]:
n = len(results)
print(f"Polytopes: {n}")
print(f"Total time: {sum(r['time'] for r in results):.1f}s")
print(f"Avg time: {sum(r['time'] for r in results)/n:.2f}s")
print()

# Phase counts
fund_phases = [r["n_fund"] for r in results]
total_phases = [r["n_total"] for r in results]
print(f"Fundamental domain phases: min={min(fund_phases)} max={max(fund_phases)} "
      f"avg={sum(fund_phases)/n:.1f}")
print(f"Total phases (after orbit): min={min(total_phases)} max={max(total_phases)} "
      f"avg={sum(total_phases)/n:.1f}")
print()

# Coxeter groups
with_orbit = [r for r in results if r["coxeter_order"] and r["coxeter_order"] > 1]
print(f"Polytopes with nontrivial Coxeter group: {len(with_orbit)}/{n}")
if with_orbit:
    type_dist = Counter(str(r["coxeter_type"]) for r in with_orbit)
    for t, c in type_dist.most_common():
        print(f"  {t}: {c} polytopes")

## Contraction type distribution

In [ ]:
# Aggregate contraction types across all polytopes
total_types = Counter()
for r in results:
    total_types.update(r["type_counts"])

print("Contraction types across all polytopes:")
for name, count in total_types.most_common():
    print(f"  {name}: {count}")

## Per-polytope detail table

In [ ]:
print(f"{'#':>3}  {'Fund':>4}  {'Total':>5}  {'Contr':>5}  {'SymFlop':>7}  "
      f"{'InfGen':>6}  {'EffGen':>6}  {'Coxeter':>12}  {'Time':>5}")
print("-" * 72)
for r in results:
    cox_str = str(r['coxeter_type']) if r['coxeter_type'] else "-"
    print(f"{r['idx']:3d}  {r['n_fund']:4d}  {r['n_total']:5d}  "
          f"{r['n_contractions']:5d}  {r['n_sym_flops']:7d}  "
          f"{r['n_inf_gens']:6d}  {r['n_eff_gens']:6d}  "
          f"{cox_str:>12}  {r['time']:5.2f}s")

## Inspect a polytope with nontrivial Coxeter group

Pick the first polytope that has symmetric flops and show its full structure.

In [ ]:
# Find first polytope with symmetric flops
sym_example = next((r for r in results if r["n_sym_flops"] > 0), None)

if sym_example:
    ekc = sym_example["ekc"]
    print(f"Polytope #{sym_example['idx']}")
    print(f"Coxeter type: {ekc.coxeter_type}")
    print(f"Coxeter group order: {ekc.coxeter_order}")
    print(f"Fundamental phases: {sym_example['n_fund']}")
    print(f"Total phases: {sym_example['n_total']}")
    print()
    
    print("Phases:")
    for phase in ekc.phases:
        print(f"  {phase.label}: int_nums shape = {phase.int_nums.shape}")
    
    print()
    print("Contractions:")
    for c in ekc.contractions:
        print(f"  {c.contraction_type.display_name()}: "
              f"curve = {c.contraction_curve}")
    
    print()
    print(f"Infinity cone generators: {ekc.infinity_cone_gens}")
    print(f"Effective cone generators: {ekc.eff_cone_gens}")
    
    if ekc.coxeter_matrix is not None:
        print(f"\nCoxeter matrix:\n{ekc.coxeter_matrix}")
else:
    print("No polytopes with symmetric flops found.")

## Inspect the most complex polytope

The one with the most phases in the fundamental domain.

In [ ]:
most_phases = max(results, key=lambda r: r["n_fund"])
ekc = most_phases["ekc"]

print(f"Polytope #{most_phases['idx']}: {most_phases['n_fund']} phases")
print()

# Graph structure
print("Phase graph:")
for phase in ekc.phases:
    neighbors = ekc.graph.neighbors(phase.label)
    print(f"  {phase.label} -> {[n.label for n in neighbors]}")

print()
print("Contraction types:")
type_counts = Counter(c.contraction_type.display_name() for c in ekc.contractions)
for name, count in type_counts.most_common():
    print(f"  {name}: {count}")